**🔹 Install dependencies**

In [ ]:
!pip install google-generativeai pypdf faiss-cpu

**🔹 Import and configure Gemini**

In [ ]:
import google.generativeai as genai
import os
from google.colab import userdata

api_key = userdata.get('gemin_api_key_2')

os.environ["GEMINI_API_KEY"] = api_key

genai.configure(api_key=os.environ["GEMINI_API_KEY"])

**Document loading (PDF + ipynb)**

**Load PDFs**

In [ ]:
from pypdf import PdfReader

folder = "/content/drive/MyDrive/QIP_IIIT_A/Learning/Projects/AI Agent Demo/Document Library"

def load_pdfs(folder):
    texts = []
    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            reader = PdfReader(os.path.join(folder, file))
            for page in reader.pages:
                texts.append(page.extract_text())
    return texts


**Load .ipynb (markdown + comments only)**

In [ ]:
import json

def load_notebooks(folder):
    texts = []
    for file in os.listdir(folder):
        if file.endswith(".ipynb"):
            with open(os.path.join(folder, file)) as f:
                nb = json.load(f)
                for cell in nb["cells"]:
                    if cell["cell_type"] == "markdown":
                        texts.append(" ".join(cell["source"]))
    return texts


**Chunking (local, cheap)**

In [ ]:
def chunk_text(texts, chunk_size=800, overlap=100):
    chunks = []
    for text in texts:
        start = 0
        while start < len(text):
            chunks.append(text[start:start+chunk_size])
            start += chunk_size - overlap
    return chunks


6️⃣ **Embeddings (IMPORTANT CHOICE)**

🔑 **To minimize Gemini calls:**

Use a **local embedding model**, not **Gemini embeddings.**

*Best options:*

*sentence-transformers/all-MiniLM-L6-v2*

**Fast, free, local**

In [ ]:
if 'ipython' not in globals():
    !pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embedder = SentenceTransformer("all-MiniLM-L6-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Create vector index** (DO THIS ONCE)

✅ Zero API calls here.

In [ ]:
pdf_texts = load_pdfs(folder)
notebook_texts = load_notebooks(folder)
all_texts = pdf_texts + notebook_texts
chunks = chunk_text(all_texts)

embeddings = embedder.encode(chunks, convert_to_numpy=True)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

**Retrieval** (local, instant)

In [ ]:
def retrieve_with_score(question, k=4):
    q_emb = embedder.encode([question])
    distances, idxs = index.search(q_emb, k)

    retrieved_chunks = [chunks[i] for i in idxs[0]]
    avg_distance = distances[0].mean()

    return retrieved_chunks, avg_distance


**Selecting Gemini Model**

In [ ]:
model = genai.GenerativeModel("gemini-2.5-flash-lite") # gemini-2.5-flash, gemini-2.5-flash-lite, gemini-3-flash-preview, gemini-3-pro-preview


**ReAct-lite Decision**

**Define a relevance threshold**

In [ ]:
RELEVANCE_THRESHOLD = 1.2  # you can tune this


In [ ]:
# def needs_retrieval(question):
#     prompt = f"""
# You are an AI assistant.
# Decide whether the following question requires
# access to workshop documents.

# Answer ONLY with YES or NO.

# Question:
# {question}
# """
#     response = model.generate_content(prompt)
#     return "YES" in response.text.upper()


**SINGLE Gemini call for answer generation**

In [ ]:
def ask_agent(question):
    retrieved_chunks, score = retrieve_with_score(question)

    if score < RELEVANCE_THRESHOLD:
        # Grounded answer
        context = "\n".join(retrieved_chunks)

        prompt = f"""
You are a workshop assistant.
If the context partially answers the question, explain cautiously.
If it does not, say "Not covered in workshop materials."

Context:
{context}

Question:
{question}
"""
        response = model.generate_content(prompt)
        return response.text

    else:
        # General answer (explicitly marked)
        prefix = (
            "Though this particular topic may not be covered in the workshop, "
            "for the purpose of this ReAct demo I’m giving a brief answer.\n\n"
        )

        prompt = f"""
Answer the following question briefly and at a high level.

Question:
{question}
"""
        response = model.generate_content(prompt)
        return prefix + response.text


In [ ]:
# def ask_agent(question):
#     if needs_retrieval(question):
#         context = retrieve_context(question)
#         prompt = f"""
# You are a workshop assistant.
# Answer primarily using the context below.
# If the context partially answers the question, explain cautiously.
# If it does not, say "Not covered in workshop materials."

# Context:
# {context}

# Question:
# {question}
# """
#         response = model.generate_content(prompt)
#         return response.text

#     else:
#         prefix = (
#             "Though this particular topic may not be covered in the workshop, "
#             "for the purpose of this ReAct demo I’m giving a brief answer.\n\n"
#         )

#         prompt = f"""Answer the following question briefly and at a high level.
# Do NOT refer to workshop materials."

# Question:
# {question}
# """
#         response = model.generate_content(prompt)
#         return prefix + response.text

#**Live demo flow**

✔ Good question (grounded)

In [ ]:
ask_agent("what is CNN")


'CNN stands for Convolutional Neural Network, also known as ConvNet or DCN. It is a multi-layer neural network characterized by local connectivity and weight sharing.'

**✔ Failure case (intentional)**

In [ ]:
ask_agent("Explain transformer attention equations in detail.")


'Though this particular topic may not be covered in the workshop, for the purpose of this ReAct demo I’m giving a brief answer.\n\nTransformer attention equations enable the model to **dynamically weigh the importance of different input elements (tokens) when processing each output element**. This is achieved through three key matrices derived from the input: **Query (Q), Key (K), and Value (V)**.\n\nHere\'s the high-level breakdown:\n\n1.  **Scoring Attention:** The core idea is to calculate how "relevant" each input token (represented by its Key vector) is to the current output token being processed (represented by its Query vector). This is done by taking the **dot product** of the Query and Key vectors. A higher dot product indicates greater relevance.\n\n    *   Equation: $Score = Q \\cdot K^T$\n\n2.  **Scaling and Softmax:** To prevent large dot products from dominating the gradients during training and to make the scores represent probabilities, the scores are **scaled** (usuall